# C3 / C4 — GPU-добор для статьи

Обучает две оставшиеся глубокие конфигурации — `C3_pure_cnn` и
`C4_pure_transformer` — на GPU. Тот же код и те же данные, что в основном
прогоне: отличается только устройство, а оно на метрику не влияет.

Встроены три стража сопоставимости — 67 признаков в том же порядке, размеры
срезов 89 973 / 11 247 и словарь ровно 498 токенов. Если хоть что-то
разойдётся с прогоном на Kaggle, ноутбук остановится с внятным сообщением,
а не выдаст молча числа, которые нельзя положить в одну таблицу с
остальными восемью конфигурациями.

## Что нужно сделать

1. **Runtime → Change runtime type → GPU** (подойдёт T4), затем **Connect**.
2. **Runtime → Run all**.
3. В конце ноутбук напечатает блок между `RESULTS_JSON_START` и
   `RESULTS_JSON_END` — **скопируй его целиком обратно в чат Claude**.

Время: примерно час-полтора на конфигурацию, итого 2–3 часа. Каждая
сохраняется сразу по готовности, поэтому обрыв сессии не уничтожает уже
досчитанное — при повторном запуске готовые конфигурации пропускаются.

In [ ]:
# 1. Проверка, что GPU действительно выделен
import torch
assert torch.cuda.is_available(), (
    "GPU не выделен! В Lightning выбери GPU-машину справа (L4); "
    "в Colab: Runtime -> Change runtime type -> GPU. Затем запусти заново.")
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# 2. Публичный код проекта + зависимости
!git clone --depth 1 -b v2-restructure https://github.com/SergeySolovyev/smart-contract-vuln-detection-from-bytecode.git repo
# kaggle пинуем на 1.8.2 -- именно эта версия работает с этим датасетом на
# машине автора. Версия из Colab по умолчанию давала 403 на приватном датасете.
!pip -q install "xgboost" "scikit-learn" pyarrow joblib "kaggle==1.8.2"
import os
print("dl_pipeline на месте:", os.path.exists("repo/src/dl_pipeline.py"))

In [ ]:
# 3. Данные: два parquet из ПУБЛИЧНОГО датасета -- ключ не нужен.
import os, urllib.request

BASE = ("https://www.kaggle.com/api/v1/datasets/download/"
        "sergeisolovyev/defi-bytecode-features-v2")
# Ожидаемые размеры: страхуют от обрыва связи и от того, что вместо файла
# прилетит HTML-страница с ошибкой.
WANT = {"train_v2.parquet": 516601748, "test_v2.parquet": 65239576}
os.makedirs("data_v2", exist_ok=True)


def fetch(name, size):
    dst = f"data_v2/{name}"
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        print(f"{name}: уже на месте")
        return
    req = urllib.request.Request(f"{BASE}/{name}",
                                 headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=300) as r, open(dst, "wb") as f:
        done = 0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if done % (50 << 20) < (1 << 20):
                print(f"  {name}: {done/1e6:.0f} / {size/1e6:.0f} MB", flush=True)
    got = os.path.getsize(dst)
    assert got == size, f"{name}: скачалось {got} байт вместо {size}"
    with open(dst, "rb") as f:
        assert f.read(4) == b"PAR1", f"{name}: это не parquet (ошибка сервера?)"
    print(f"{name}: готов, {got/1e6:.0f} MB")


for n, s in WANT.items():
    fetch(n, s)
print("данные готовы")

In [ ]:
# 4. Обучение C3 и C4 на GPU.
#    Токенизация ПОТОКОМ: строки байткода (8.8 ГБ) никогда не лежат в памяти
#    целиком -- иначе пик 12.4 ГБ и бесплатный Colab падает по ОЗУ.
import sys, gc, time, json, pathlib
import numpy as np, pandas as pd, pyarrow.parquet as pq, torch
sys.path.insert(0, "repo/src")
from dl_pipeline import (BytecodeTokenizer, DL_EXPERIMENTS, run_dl_experiment)

LABELS = ["access-control", "arithmetic", "bad-randomness", "double-spending",
          "locked-ether", "other", "reentrancy", "unchecked-calls"]
FEATURES = json.load(open("repo/data/feature_columns.json"))
WANT = {"C3_pure_cnn", "C4_pure_transformer"}
MAX_LEN = 20000

# Стражи сопоставимости с восемью конфигурациями, обученными на Kaggle.
assert len(FEATURES) == 67, f"ожидалось 67 признаков, получено {len(FEATURES)}"


def small(path):
    """Признаки и метки -- их немного (~25 МБ), берём как есть."""
    df = pd.read_parquet(path, columns=FEATURES + LABELS)
    return (df[FEATURES].to_numpy("float32"), df[LABELS].to_numpy("float32"))


def fit_tokenizer(path, n=20000):
    """Ровно первые 20000 строк, как в прогоне на Kaggle."""
    rows, f = [], pq.ParquetFile(path)
    for b in f.iter_batches(batch_size=4096, columns=["bytecode"]):
        rows.extend(b.column("bytecode").to_pylist())
        if len(rows) >= n:
            break
    tok = BytecodeTokenizer().fit(rows[:n])
    del rows, f
    gc.collect()
    return tok


def encode_stream(path, tok, total):
    """Кодируем побатчево; строки каждого батча сразу освобождаются."""
    out, done, f = [], 0, pq.ParquetFile(path)
    for b in f.iter_batches(batch_size=2048, columns=["bytecode"]):
        chunk = b.column("bytecode").to_pylist()
        out.extend(tok.encode_unpadded(s, MAX_LEN) for s in chunk)
        done += len(chunk)
        del chunk, b
        if done % 20480 == 0:
            gc.collect()
            print(f"    {done}/{total}", flush=True)
    del f
    gc.collect()
    return out


TR, TE = "data_v2/train_v2.parquet", "data_v2/test_v2.parquet"
n_tr = pq.ParquetFile(TR).metadata.num_rows
n_te = pq.ParquetFile(TE).metadata.num_rows
print(f"train {n_tr:,}  test {n_te:,}", flush=True)
assert n_tr == 89973 and n_te == 11247, "неожиданные размеры срезов"

X_tr, y_tr = small(TR)
X_te, y_te = small(TE)

tok = fit_tokenizer(TR)
print("vocab:", tok.vocab_size, flush=True)
assert tok.vocab_size == 498, (
    f"словарь {tok.vocab_size} вместо 498 -- входы отличаются от прогона на "
    "Kaggle, числа были бы несопоставимы. Останов.")

print("кодирую train ...", flush=True)
tr_ids = encode_stream(TR, tok, n_tr)
print("кодирую test ...", flush=True)
te_ids = encode_stream(TE, tok, n_te)
gc.collect()

RUNS = pathlib.Path("runs_out"); RUNS.mkdir(exist_ok=True)
out = {}
for cfg in DL_EXPERIMENTS:
    if cfg["name"] not in WANT:
        continue
    done_file = RUNS / f"{cfg['name']}.json"
    if done_file.exists():                      # переживает обрыв сессии
        out[cfg["name"]] = json.loads(done_file.read_text())
        print(f"{cfg['name']}: уже посчитан, пропускаю", flush=True)
        continue
    print(f"\n=== {cfg['name']} ===", flush=True)
    t0 = time.time()
    res = run_dl_experiment(cfg, tok, tr_ids, y_tr, X_tr, te_ids, y_te, X_te,
                            runs_dir=RUNS, wandb_run=None)
    done_file.write_text(json.dumps(res))
    mf = res.get("macro_f1_external", res.get("macro_f1"))
    if mf is None:
        mf = float(np.mean(res["f1_per_label"]))
    out[cfg["name"]] = res
    print(f"  {cfg['name']} macro_f1={mf:.4f} "
          f"({(time.time()-t0)/60:.0f} мин)", flush=True)
    torch.cuda.empty_cache(); gc.collect()
print("\nобе конфигурации обучены")

In [ ]:
# 5. Отдать результат: JSON для копипаста в чат + сохранённые файлы
import json
for name, res in out.items():
    open(f"{name}.json", "w").write(json.dumps(res))
print("\n=========== СКОПИРУЙ ВСЁ МЕЖДУ МЕТКАМИ ОБРАТНО В ЧАТ CLAUDE ===========\n")
print("RESULTS_JSON_START")
print(json.dumps(out))
print("RESULTS_JSON_END")
print("\nТакже сохранены файлы:", [f"{n}.json" for n in out],
      "\n(их можно скачать из файлового браузера Studio, но хватит и JSON выше)")